In [24]:
import pandas as pd

df = pd.read_parquet("assets/merged.parquet")
print(df.head(1))

   example_id input_text  input_tokens output_text  output_tokens  new_tokens  \
0         NaN          a           1.0           x            1.0         0.0   

   inference_time_s  tokens_per_second  gpu_num_samples  gpu_duration_ms  ...  \
0               0.1                5.0              1.0            100.0  ...   

   gpu_power_min_w  gpu_memory_avg_mib  gpu_memory_max_mib  gpu_gpu_util_avg  \
0             50.0              1000.0              1000.0              10.0   

   gpu_gpu_util_max  gpu_temp_avg_c  gpu_temp_max_c  gpu_energy_j  test_col  \
0              10.0            30.0            30.0           5.0       NaN   

   message  
0      NaN  

[1 rows x 22 columns]


In [25]:
from ml_data import df_to_records
from schemas import InferenceRecord
import random

records = df_to_records(df)
if not records:
    print("No records found in dataframe")
else:
    dropped = len(df) - len(records)
    print(f"Converted {len(records)} rows into {InferenceRecord.__name__} instances")
    if dropped:
        print(f"({dropped} rows were skipped due to missing identifiers)")
    print("Sample random record:\n", random.choice(records))

Converted 1827 rows into InferenceRecord instances
(7 rows were skipped due to missing identifiers)
Sample random record:
 InferenceRecord(example_id=324, input_text='Let $n$ be a positive integer, and define $S_{n}=\\{1,2, \\ldots, n\\}$. Consider a non-empty subset $T$ of $S_{n}$. We say that $T$ is balanced if the median of $T$ is equal to the average of $T$. For example, for $n=9$, each of the subsets $\\{7\\},\\{2,5\\},\\{2,3,4\\},\\{5,6,8,9\\}$, and $\\{1,4,5,7,8\\}$ is balanced; however, the subsets $\\{2,4,5\\}$ and $\\{1,2,3,5\\}$ are not balanced. For each $n \\geq 1$, prove that the number of balanced subsets of $S_{n}$ is odd.\n\n(To define the median of a set of $k$ numbers, first put the numbers in increasing order; then the median is the middle number if $k$ is odd, and the average of the two middle numbers if $k$ is even. For example, the median of $\\{1,3,4,8,9\\}$ is 4 , and the median of $\\{1,3,4,7,8,9\\}$ is $(4+7) / 2=5.5$. $)$', input_tokens=399, output_text='ana

In [26]:
# Train in-notebook using the shared library functions
import importlib
import ml_model
importlib.reload(ml_model)
from ml_model import train_and_evaluate, save_model

pipe, sample_df = train_and_evaluate(df, test_size=0.2, random_state=0)
# persist the trained pipeline next to the notebook
save_model(pipe, "models/model.joblib")
print("Saved model to models/model.joblib — sample predictions:")
print(sample_df[["input_text", "input_tokens", "pred_output_tokens", "pred_gpu_energy_j"]].head())


Dropped 6 rows with missing values before training.
output_tokens: RMSE=14.6116, R2=0.9213
gpu_energy_j: RMSE=53.7232, R2=0.8468
Saved model to models/model.joblib — sample predictions:
                                          input_text  input_tokens  \
0  Copy all files matching "*failed.ipynb" in the...         127.0   
1  Assume that all angles of a triangle $A B C$ a...         219.0   
2  Write a Python script at /home/user/Desktop/co...         152.0   
3  Let $n$ be an integer greater than 1 , and let...         219.0   
4  18. 6b.(CAN 5) Let $x_{1}, x_{2}, \ldots, x_{n...         252.0   

   pred_output_tokens  pred_gpu_energy_j  
0          173.851786         497.613301  
1          269.871525         585.643472  
2          198.690615         547.197403  
3          269.540649         578.279403  
4          293.915119         597.224708  
